In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [ ]:
import os

# --- PATH CONFIGURATION ---
# Set this to True if you are running in Google Colab
RUNNING_IN_COLAB = True

if RUNNING_IN_COLAB:
    # By default, Colab uploads go to the current directory (/content/)
    TRAIN_PATH = 'train_dataset.csv'
    TEST_PATH = 'test_dataset.csv'
    PREDS_DIR = '.'
else:
    # Default local paths
    TRAIN_PATH = '../datasets/train_dataset.csv'
    TEST_PATH = '../datasets/test_dataset.csv'
    PREDS_DIR = '../predictions'

os.makedirs(PREDS_DIR, exist_ok=True)

# RandomForest Regression
This notebook documents the RandomForest implementation for predicting the continuous flow reactor's overall yield, utilizing GridSearchCV for tuning tree depth and leaf samples.

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
train_df['residence_proxy'] = train_df['length_m'] / train_df['flow_rate_L_min']
test_df['residence_proxy'] = test_df['length_m'] / test_df['flow_rate_L_min']

train_df['mean_T'] = (train_df['inlet_temperature_K'] + train_df['jacket_temperature_K']) / 2
test_df['mean_T'] = (test_df['inlet_temperature_K'] + test_df['jacket_temperature_K']) / 2

features = [
    'flow_rate_L_min',
    'concentration_mol_L',
    'inlet_temperature_K',
    'length_m',
    'jacket_temperature_K',
    'residence_proxy',
    'mean_T'
]
target = 'overall_yield'

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=42, n_jobs=-1))
])

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    'rf__n_estimators': [100, 200, 300],
    'rf__max_depth': [None, 5, 10, 15],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(pipeline, param_grid, scoring='neg_root_mean_squared_error', cv=kf, n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV RMSE (raw): {-grid_search.best_score_:.4f}")

In [ ]:
fold_rmses = []
fold_rmses_clipped = []

for train_idx, val_idx in kf.split(X_train):
    X_tr_f, X_val_f = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr_f, y_val_f = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    pipeline.set_params(**grid_search.best_params_)
    pipeline.fit(X_tr_f, y_tr_f)
    
    val_preds = pipeline.predict(X_val_f)
    val_preds_clipped = np.clip(val_preds, 0, 100)
    
    fold_rmses.append(np.sqrt(mean_squared_error(y_val_f, val_preds)))
    fold_rmses_clipped.append(np.sqrt(mean_squared_error(y_val_f, val_preds_clipped)))

raw_oof_rmse = np.mean(fold_rmses)
clipped_oof_rmse = np.mean(fold_rmses_clipped)
mean_fold_rmse = clipped_oof_rmse
std_fold_rmse = np.std(fold_rmses_clipped, ddof=1)

print(f"Raw RMSE = {raw_oof_rmse:.4f}")
print(f"Clipped RMSE = {clipped_oof_rmse:.4f}")
print(f"Clipping improvement = {raw_oof_rmse - clipped_oof_rmse:.4f}")

print("\nClipped fold RMSEs:")
for rmse in fold_rmses_clipped:
    print(f"{rmse:.4f}")

print(f"\nMean CV RMSE = {mean_fold_rmse:.4f}")
print(f"Standard deviation = {std_fold_rmse:.4f}")

In [ ]:
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

preds = np.clip(best_model.predict(X_test), 0, 100)
output_path = os.path.join(PREDS_DIR, 'RandomForest_predictions.csv')
pd.DataFrame({'overall_yield': preds}).to_csv(output_path, index=False)
print(f"Predictions saved to {output_path}")

## Visualizations
The following plots help visualize the model's behavior and are highly recommended for the Phase 2 pitch.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(8, 6))

y_train_pred = np.clip(best_model.predict(X_train), 0, 100)
plt.scatter(y_train, y_train_pred, alpha=0.6, color='g', edgecolor='w')
plt.plot([0, 100], [0, 100], 'r--', lw=2) 
plt.title('RandomForest Training Data: True vs Predicted Yield', fontsize=14, pad=15)
plt.xlabel('True Yield (%)')
plt.ylabel('Predicted Yield (%)')

plt.tight_layout()
plt.show()